# 2.2.3 年龄区间分类

问卷里没有“日常运动量”字段。原标题写运动量，正文写预测年龄；`Your age` 实际是年龄区间。

本题预测区间类别。不要把 `19 to 25` 取成整数 19，也不要做回归或报告 R²。验证集只用来比较分类器，测试集只看一次。准确率容易被人数最多的区间抬高，请同时看宏平均 F1 和混淆矩阵。

In [ ]:
import json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.base import clone

def prepare(X):
    numeric = X.select_dtypes(include="number").columns.tolist()
    categorical = [c for c in X.columns if c not in numeric]
    return ColumnTransformer([
        ("number", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
        ("category", Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])

df = pd.read_csv("fitness analysis.csv")
df.columns = df.columns.str.strip()
target = "Your age"
features = ["Your gender", "How important is exercise to you ?", "How healthy do you consider yourself?"]
df = df.dropna(subset=[target, *features]).copy()
print(df[target].value_counts())
y = df[target].astype(str)  # 保留区间文字
X = df[features]

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev)

candidates = {
    # 空白处应是分类器，不要写成 RandomForestRegressor。
    "random_forest": __________(n_estimators=100, max_depth=6, random_state=42, n_jobs=2),
    "logistic": LogisticRegression(max_iter=1500),
}
validation = {}
for name, estimator in candidates.items():
    model = Pipeline([("prepare", prepare(X_train)), ("model", estimator)])
    model.fit(X_train, y_train)
    validation[name] = float(f1_score(y_valid, model.predict(X_valid), average="macro", zero_division=0))
chosen = max(validation, key=validation.get)
final_model = Pipeline([("prepare", prepare(X_dev)), ("model", clone(candidates[chosen]))])
final_model.fit(X_dev, y_dev)
prediction = final_model.predict(X_test)
labels = final_model.classes_.tolist()
metrics = {
    "task": "age-interval-classification",
    "chosen_model": chosen,
    "validation_macro_f1": validation,
    "accuracy": float(accuracy_score(y_test, prediction)),
    "macro_f1": float(__________(y_test, prediction, average="macro", zero_division=0)),
    "labels": labels,
    "confusion_matrix": confusion_matrix(y_test, prediction, labels=labels).tolist(),
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_valid)),
    "test_rows": int(len(X_test)),
}
Path("2.2.3_metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print(metrics["chosen_model"], round(metrics["accuracy"], 4), round(metrics["macro_f1"], 4))
